# 02 - Cross-validated model training

Reproduces the validation methodology used by `ml/src/train.py` and explains the
three decisions behind it:

1. **preprocessing inside the folds** - the `ColumnTransformer` is refit on every
   training fold, so no test-fold statistic ever informs its own transform;
2. **nested CV for thresholds** - each target's decision threshold is chosen on
   *inner* folds, so the reported tuned metrics are not scored on the data that
   selected the threshold;
3. **calibration as a separate monotonic map** fit on out-of-fold scores, which
   keeps a single LightGBM booster as the object that is both served and explained.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "ml" / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))
DATA = ROOT / "ml" / "data" / "z_alizadeh_sani.xlsx"
MODELS = ROOT / "backend" / "app" / "models"
RESULTS = ROOT / "ml" / "results"
print("repo root:", ROOT)

In [ ]:
from train import TARGETS, VESSELS, build_pipeline, extract_targets, load_dataset, nested_cv

df = load_dataset(DATA)
X, y = extract_targets(df)
X.shape

## The leakage guard

`extract_targets` drops `CAD`, `Cath`, `LAD`, `LCX` and `RCA` and asserts none of
them survive into `X` - requirement 1d, enforced in code rather than by convention.

In [ ]:
LEAK = {"CAD", "Cath", "LAD", "LCX", "RCA"}
assert not LEAK & set(X.columns), "leakage!"
print(f"{X.shape[1]} features, none of {sorted(LEAK)} present")
print("\nhighest absolute correlation with each target:")
num = X.select_dtypes("number")
for t in TARGETS:
    corr = num.corrwith(pd.Series(y[t], index=num.index)).abs().sort_values(ascending=False)
    print(f"  {t:4s} " + ", ".join(f"{c}={v:.2f}" for c, v in corr.head(3).items()))

Nothing approaches the near-1.0 correlation that a leaked target would produce.

## Why preprocessing must sit inside the pipeline

The original script called `fit_transform` on all 303 rows *before* splitting. The
leak is label-independent, so its effect is small, but it is not defensible under
review and a `Pipeline` removes it for free.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

pipe = build_pipeline(X, "CAD", seed=42)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y["CAD"], cv=cv, scoring="roc_auc")
print(f"CAD ROC-AUC, preprocessing inside folds: {scores.mean():.4f} +/- {scores.std():.4f}")
print("per fold:", np.round(scores, 4))

## Nested cross-validation for all four targets

This is the exact call `train.py` makes.

In [ ]:
summaries = {}
for target in TARGETS:
    summaries[target] = nested_cv(X, y[target], target, folds=5, seed=42)["summary"]

## Reading the results

PR-AUC matters more than ROC-AUC for LCX and RCA, whose positive class is the
minority. The gap between the 0.5 row and the tuned row is the whole argument for
thresholding: at 0.5 the vessel models are too conservative to be useful for a
screening-style decision-support tool.

In [ ]:
rows = []
for t in TARGETS:
    s = summaries[t]
    rows.append({
        "target": t,
        "prevalence": s["positive_prevalence"],
        "roc_auc": s["threshold_independent"]["roc_auc"],
        "pr_auc": s["threshold_independent"]["pr_auc"],
        "recall@0.5": s["at_threshold_0.5"]["recall"],
        "recall@tuned": s["at_tuned_threshold"]["recall"],
        "f1@tuned": s["at_tuned_threshold"]["f1"],
        "threshold": s["tuned_threshold_mean"],
    })
pd.DataFrame(rows).set_index("target")

## Class weighting

`class_weight="balanced"` is applied to **LCX only**. Both LCX and RCA have a
minority positive class, so reweighting looks like it should help both -- but under
nested CV it improves LCX marginally (ROC-AUC 0.736 -> 0.742) and makes RCA *worse*
on every metric (ROC-AUC 0.698 -> 0.694, recall at the tuned threshold 0.745 ->
0.685). Every one of those gaps is well inside one standard deviation, so treat this
as a weak preference rather than a finding.

The quick check below uses plain (non-nested) `cross_val_predict`, which is cheaper
but noisier and can disagree with the nested result above -- a useful reminder that
at N=303 these comparisons are fragile. **The recall improvement on LCX and RCA comes
overwhelmingly from threshold tuning, not from class weighting.**

In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.model_selection import cross_val_predict

for target in ["LCX", "RCA"]:
    for weight in [None, "balanced"]:
        p = build_pipeline(X, target, seed=42)
        p.set_params(clf__class_weight=weight)
        prob = cross_val_predict(p, X, y[target], cv=cv, method="predict_proba")[:, 1]
        ap = average_precision_score(y[target], prob)
        print(f"{target} class_weight={str(weight):9s} PR-AUC={ap:.4f}")

Run `python ml/src/train.py` to persist the artifacts, then
`python ml/src/evaluate.py` for the ROC / PR / calibration / confusion figures.